## Intercomparison of inverse modelling results

This notebook contains options to plot and compare results from different inverse models. The following variables can be investigated:

- Posterior and prior country fluxes, total from all sectors, and respective uncertainties
- Observed and posterior/prior modelled total mole fractions
- Posterior and prior modelled baseline mole fractions
- Posterior and prior bias mole fractions
- Posterior and prior mole fractions from the outer regions
- Observed repeatability/variability, model and total uncertainties
- Posterior and prior spatial fluxes

Future updates may include:

- Sector-level emissions
- Comparison between each model's country/region definition

### Input data:

- Flux and concentration netCDF files with model outputs
- json files with models, sites and species information
- (optional) netCDF file with country fluxes from bottom-up inventory
- (optional) netCDF file with baseline timestamps

Please refer to the User Guide for information on files format, filename convention and folder structure.

### Notebook setup:

1. Decide the verbosity level by assigning the logging level to `logging.INFO` or `logging.WARNING`.

2. Edit the `data_dir` to point towards where the model output is.

3. Set `presentation_mode` to `True`/`False` to use big font sizes (recommended for slides) or small font sizes (recommended for text documents). 

4. Update the `experiments` dictionary so that each key points towards the models you want to plot.

5. Run the cell below, before running any of the plotting code.

In [ ]:
%load_ext autoreload
%autoreload 2
import logging
logging.basicConfig(level=logging.INFO)
import xarray as xr
import numpy as np

from fluxie.config import set_print_settings
from fluxie.io import read_config_files

### Path to results directory 
data_dir = 'ENTER FULL PATH TO RESULTS DIRECTORY'

### Set presentation_mode to True for bigger fonts
presentation_mode = False

### Group the models of interest in meaningful experiment names 
# For figures combining results include both models
# For spatial map figures with only one model result only select one model
# This cell indicates the experiements for all subsequent cells so needs to be run first
experiments = {'base_results'              :[
                                 'rhime_name_CENTRALASIA_flat_base',
                                #  'InTEM_NAME_EASTASIA_flat_base', 
                                ]
              }

###################################

config_data = read_config_files("../configs/")
annotate_coords = set_print_settings(presentation_mode)

### 1. Timeseries of country/region fluxes

##### Edit and run this cell to read and pre-process the data:

In [ ]:
from fluxie.io import read_model_output, read_flux_total_fgases
from fluxie.operators.select import slice_flux
from fluxie.config import set_model_colors
from fluxie.config import set_model_labels
from fluxie.operators.flux_scale_by_sector_proportions import scale_by_sector_proportions

###################################
### edit variables in this block
species = 'cf4' #options for individual species, or 'all_hfc' or 'all_pfc'
models = experiments['base_results'] #list of model names to include in the plot. Must match names in models_info.json. If using 'all' for species, this will be ignored and all models with that species will be included.
# models = experiments['sensitivity'] #list of model names to include in the plot. Must match names in models_info.json. If using 'all' for species, this will be ignored and all models with that species will be included.

regions = ['INDIA','INDIA-NSE','INDIA-NORTH', 'INDIA-SOUTH', 'INDIA-EAST', 'INDIA-WEST'] #if the model outputs are available at this spatial resolution, otherwise it will be aggregated to the larger regions.
period='yearly'
country_flux_units_print = 'Gg yr-1' #desired units for the plot. Add "CO2-eq" to convert the mass to CO2 equivalent (e.g. "Gg CO2-eq yr-1")
start_date = '2023-01-01' #inclusive. Option to set as list of dates, e.g. ['2018-01-01','2019-01-01'] which is required for total fgases if one model is missing obs for a year
end_date = '2026-03-01' #not inclusive. Option to set as list of dates, e.g. ['2023-01-01','2022-01-01'] which is required for total fgases if one model is missing obs for a year
get_labels_from_file = True # If False, uses default labels. If True, uses labels in models_info.json.
sector_file = None  # if not None, adds sector level results by scaling model totals by the sector proportions in this file. Default file: 'EUROPE_EDGAR', inside the 'sector_flux' directory
create_region_sector_totals = True #if True, uses country_fraction variable to sum spatial sector fluxes to region sector fluxes
read_standard_run = False
only_overlapping = True #if True, only uses time periods where all models have data
###################################

ds_all_flux_scaled = {}

if 'all' in species:
    ds_all_flux_scaled = read_flux_total_fgases(data_dir,species,models,config_data,
                                                     regions,start_date,end_date,
                                                     period=period)
else:
    ds_all_flux = read_model_output(data_dir,"flux",species,models,config_data,period=period)

    for m in models:
        ds_all_flux_scaled[m] = slice_flux({m:ds_all_flux[m]},config_data,start_date,end_date,species=species,country_flux_units_print=country_flux_units_print)[m]

if sector_file is not None:
    ds_all_flux_scaled = scale_by_sector_proportions(data_dir=data_dir,ds_all=ds_all_flux_scaled,
                                                            species=species,
                                                            country_flux_units_print=country_flux_units_print,
                                                            config_data=config_data,regions=regions,
                                                            sector_file=sector_file,create_region_sector_totals=create_region_sector_totals)   

model_colors = set_model_colors(models)
model_labels = set_model_labels(models,config_data,get_labels_from_file)

#### Timeseries of country fluxes:

In [ ]:
from fluxie.plots.flux_timeseries import plot_country_flux

###################################
### edit variables in this block
plot_inventory = False
plot_inventory_uncertainty = False    #Either True/False or a list of True/False to match length of inventory_years
inventory_years = None #If None, plots most recent. Or can choose list of years: ['2022','2023']
inventory_filename = 'UNEP_inventory'   #Full filename = {inventory_filename}_{species}_{inventory_year}
fix_y_axes = False #if True: all y axis limits are the same, if False: each y axis is relative to the data
                     #if a list of floats (e.g. [0,0.1]) applies these limit to all axes
add_prior = True #if True: plots prior as dashed lines
add_prior_unc = False #if True: plots prior uncertainty as shaded area
set_global_leg = True #If True, plots one single legend instead of one legend per subplot.
country_codes_as_titles = False #If True, lists 3-letter country codes under region names in subplot titles. Set to None for no title.
plot_separate = True #If True, includes all model results as separate lines (or insert a list of boolean of the same length as models to specify which models to plot)
plot_separate_unc = True #If True, includes all model uncertainty as shaded areas
plot_combined = False #If True, combined results, averaged from all models (or insert a list of boolean of the same length as models to specify which models to combine)
combined_models_dict = None #To plot multiple combined results, use (e.g.) combined_models_dict = {'NAME Mean': ['InTEM_NAME', 'ELRIS_NAME'], 'FLEXPART Mean': ['InTEM_FLEXPART', 'ELRIS_FLEXPART']} and set plot_combined = True.
plot_combined_unc = True #If True, includes combined uncertainty as shaded area
resample = None #If None, no resample is done. Else resample the data to the given period (options 'year' and 'season' for yearly and seasonal averages)
resample_uncert_correlation = False #If True, uses mean uncertainty during resampling, if False, recalculates uncertainty assuming no correlation.
plot_resample_and_original = False #If True, plots both the resampled and original data
annex_mode = False #If True, replace the labels with more concise versions for NID Annexes.
rolling_mean = False # If True, calculates a rolling mean of the data (insert a list of boolean of the same length as models to specify the models to smooth)
sector = 'total'     # default 'total', otherwise chosen single sector plotted for all models
aggreg_month = False     # if True, plot the fluxes aggregated by month. Only work with monthly data.
xticks_at_centre = False    # if True, moves x ticks to centre of time periods
plot_grid = True    # if True, adds faint grid to the plots
add_vline = None # List of dates (str) to add vertical lines to the plots, e.g. ['2016-07-02', '2023-07-02']
secondary_units = None # If not None, e.g. 'Gg yr-1', adds a secondary y axis with these units
###################################

fig, _ = plot_country_flux(ds_all_flux_scaled,
                        species,
                        regions,
                        config_data,
                        model_colors,
                        model_labels,
                        start_date,
                        end_date,
                        annex_mode,
                        plot_inventory,
                        plot_inventory_uncertainty,
                        inventory_years,
                        inventory_filename,
                        data_dir,
                        fix_y_axes,
                        add_prior,
                        add_prior_unc,
                        set_global_leg,
                        country_codes_as_titles=country_codes_as_titles,
                        plot_separate=plot_separate,
                        plot_separate_unc=plot_separate_unc,
                        plot_combined=plot_combined,
                        combined_models_dict = combined_models_dict,
                        plot_combined_unc=plot_combined_unc,
                        resample=resample,
                        resample_uncert_correlation=resample_uncert_correlation,
                        plot_resample_and_original=plot_resample_and_original,
                        rolling_mean=rolling_mean,
                        aggreg_month=aggreg_month,
                        xticks_at_centre=xticks_at_centre,
                        plot_grid=plot_grid,
                        add_vline=add_vline,
                        secondary_units=secondary_units,
                        return_res=True,sector=sector)

Save plot here:

In [ ]:
# output_path = f'PATH'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

### 2. Modelled and observed mole fractions and/or baselines

##### Edit and run this cell to choose inputs and read in the data:

In [ ]:
from fluxie.io import read_model_output
from fluxie.operators.select import slice_mf
from fluxie.config import set_model_colors
from fluxie.config import set_model_labels

###################################
### edit variables in this block
species = 'cf4' 
site = 'MCOH'
intake_height = None
models = experiments['base_results'] #list of model names to include in the plot. Must match names in models_info.json. If using 'all' for species, this will be ignored and all models with that species will be included.
period = 'yearly'
mf_units_print = 'ppt'
start_date = '2023-01-01' #inclusive
end_date = '2026-03-01'   #not inclusive
baseline_site = None #'MHD', 'JFJ' or 'CMN'. If None, does not mask by baseline time
baseline_filename = 'InTEM_baseline_timestamps'
get_labels_from_file = True # If False, uses default labels. If True, uses labels from config file.
read_standard_run = False
###################################

ds_all_mf = read_model_output(data_dir,'concentration',species,models,config_data,period=period,read_standard_run=read_standard_run)

ds_all_mf_sliced = slice_mf(ds_all_mf.copy(),start_date,end_date,site,baseline_site=baseline_site,
                              baseline_filename=baseline_filename,data_dir=data_dir,
                              mf_units_print=mf_units_print,intake_height=intake_height)

model_colors = set_model_colors(models)
model_labels = set_model_labels(models,config_data,get_labels_from_file)

#### Plot timestamps with data at each site:

In [ ]:
from fluxie.plots.mf_timeseries import plot_sites_timeseries

separate_by_height = False      #If True, plots data split by site and by height

fig = plot_sites_timeseries(ds_all_mf,'mf_posterior',species,start_date,end_date,model_colors,model_labels,config_data,margin=0.2,
                            separate_by_height=separate_by_height)

Save plot:

In [ ]:
# output_path = 'PATH'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

#### Timeseries plot, separated by model:

In [ ]:
from fluxie.plots.mf_timeseries import plot_timeseries

###################################
### edit variables in this block
# Variables and respective uncertainties to plot
include = {
            # 'stdev_mf_model'  : None,
           #'mf_bc_posterior' : None,
           #'mf_bc_prior' : None,
           'mf_posterior' : "percentile_mf_posterior",
            'mf_observed': None 
           }

# To plot the histogram of the variables in "include", set "diff_include" to None
# To plot the histogram of Obs-variable, set "diff_include" to the desired variable to be subtracted
diff_include = ['mf_posterior']

y_lim = None # to choose y-axis limits set y_lim=[min_value,max_value]
aggreg_month = False # set to true to aggregate mf data by month and see the seasonnal cycle
###################################

###################################
### options for variables to include
# mf_observed         - total observed mole fraction
# mf_prior            - prior total mole fraction
# mf_posterior        - posterior total mole fraction
# mf_bc_prior         - prior baseline
# mf_bc_posterior     - posterior baseline
# mf_bias_prior       - prior bias added to site
# mf_bias_posterior   - posterior bias added to site
# mf_outer_prior      - prior mole fractions only from outer regions
# mf_outer_posterior  - posterior mole fractions only from outer regions
# stdev_mf_observed_repeatability - observed repeatability mole fraction uncertainty
# stdev_mf_observed_variability   - observed variability mole fraction uncertainty
# stdev_mf_model                  - model mole fraction uncertainty
# stdev_mf_total                  - total mole fraction uncertainty

### options for uncertainties to plot as error bars/shaded area
# stdev_mf_observed_repeatability - observed repeatability mole fraction uncertainty
# stdev_mf_observed_variability   - observed variability mole fraction uncertainty
# stdev_mf_model                  - model mole fraction uncertainty
# stdev_mf_total                  - total mole fraction uncertainty
# percentile_mf_prior             - prior mole fraction uncertainty
# percentile_mf_posterior         - posterior mole fraction uncertainty
###################################

fig = plot_timeseries(ds_all_mf_sliced,include,species,site,
                                    model_colors,model_labels,config_data,annotate_coords,presentation_mode,
                                    plot_type='separate',
                                    diff_include=diff_include,
                                    y_lim=y_lim,intake_height=intake_height,aggreg_month=aggreg_month)

Save plot:

In [ ]:
# output_path = 'PATH'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

#### Timeseries plot, all models together:

In [ ]:
from fluxie.plots.mf_timeseries import plot_timeseries

###################################
### edit variables in this block (see instructions above)
include = {'mf_posterior' : 'percentile_mf_posterior'}
include = {'stdev_mf_total' : None}
diff_include=['mf_posterior']
y_lim=None
###################################

fig = plot_timeseries(ds_all_mf_sliced,include,species,site,
                                    model_colors,model_labels,config_data,annotate_coords,presentation_mode,
                                    plot_type='together',
                                    diff_include=diff_include,
                                    y_lim=y_lim,intake_height=intake_height)

Save plot:

In [ ]:
# output_path = 'PATH'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

#### Timeseries plot, separated by model, all sites together:

In [ ]:
from fluxie.operators.select import slice_mf
from fluxie.plots.mf_timeseries import plot_sites_list_mf

###################################
### edit slicing options in this block
sites = ['MCOH','BCOB'] 
intake_height = None
mf_units_print = 'ppt'
start_date = '2023-01-01' #inclusive
end_date = '2026-03-01'   #not inclusive
baseline_site = None #'MHD', 'JFJ' or 'CMN'. If None, does not mask by baseline time
baseline_filename = 'InTEM_baseline_timestamps'
###################################

### edit plotting options in this block
include = {'mf_observed'   : None,
           'mf_posterior' : 'percentile_mf_posterior'
           }

aggreg_month = False
data_on_single_graph="sites" # if "models", plots all models on the same graph. If "sites", plots all sites on the same graph.
uncertainty_type="FillBetween"
###################################

ds_all_mf_sliced_multisites = slice_mf(ds_all_mf.copy(),start_date,end_date,sites,baseline_site=baseline_site,
                            baseline_filename=baseline_filename,data_dir=data_dir,
                            mf_units_print=mf_units_print,intake_height=intake_height)

fig = plot_sites_list_mf(ds_all_mf_sliced_multisites,
                         sites,
                         species,
                         include,
                         model_labels,
                         model_colors,
                         aggreg_month,
                         config_data,
                         data_on_single_graph,
                         unc_type=uncertainty_type)




Save plot:

In [ ]:
# output_path = 'PATH'

# fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

#### Calculate and plot stats for the fit to obs:

In [ ]:
from fluxie.operators.select import slice_mf
from fluxie.operators.mf import stats_mf
from fluxie.plots.mf_stats import plot_stats_mf


###################################
### edit variables in this block
stats_to_plot = ['pearson', 'bias', 'crmse']
what_to_compare = 'posterior_above_BC'
stats_ylim = None    # or set it to None
get_labels_from_file = True # If False, uses default labels. If True, uses labels from config file.


### Implemented statistics (stats_to_plot)
# pearson - Pearson correlation coefficient
# bias    - Mean difference
# rmse    - RMSE
# nrmse   - Normalised RMSE (division by mean observation)
# crmse   - Centered (bias-corrected) RMSE
# std_obs  - Standard deviation of observations 
# std_sim  - Standard deviation of simulation 
# std_res  - Standard deviation of residuals
# nn      - Number of observation-simulation pairs 
###################################

ds_all_allsites = slice_mf(ds_all_mf.copy(), 
                        start_date,
                        end_date,site=None,
                        baseline_site=baseline_site,
                        baseline_filename=baseline_filename,
                        data_dir=data_dir,
                        mf_units_print=mf_units_print)

model_labels = set_model_labels(models,config_data,get_labels_from_file)

stats = stats_mf(ds_all_allsites, stats_type=what_to_compare)

fig = plot_stats_mf(stats=stats, 
                    stats_to_plot=stats_to_plot, 
                    species=species,
                    model_colors=model_colors, 
                    model_labels=model_labels, 
                    config_data=config_data, 
                    mf_units_print=mf_units_print,
                    stats_type=what_to_compare,
                    stats_ylim=stats_ylim,
                    start_date=start_date, 
                    end_date=end_date)

#### Plot Taylor Diagrams

In [ ]:
from fluxie.operators.mf import stats_mf
from fluxie.plots.mf_stats import plot_taylor_diagram

###################################
### edit variables in this block
include = ['prior','posterior'] # Variables to include in the Taylor diagram
plot_type_model = 'together' #  If 'separate', plots each model in a separate subplot. If 'together', plots all models in the same subplot.
plot_type_stat = 'together'  # If 'separate', plots each statistic in a separate subplot. If 'together', plots all statistics in the same subplot.
normalize = True # If True, normalizes the data by the standard deviation of the observations. If False, plots the absolute data.
stat_markers = ['s'] # Marker styles used to plot the stats specified in the `include` parameter, sharing the same index. 
std_range = [0, 1.5] # Range for the standard deviation axis, in units provided with `std_unit` if `normalize` = True.
check_sites = False # If True, color different sites with different colors
###################################

model_labels = set_model_labels(models, config_data, get_labels_from_file)

stats = {}
for stat in include:
    stats[stat] = stats_mf(ds_all_allsites, stats_type=stat)

fig = plot_taylor_diagram(
    stats=stats,
    model_colors=model_colors,
    model_labels=model_labels,
    stat_markers=stat_markers,
    normalize=normalize,
    plot_type_model=plot_type_model,
    plot_type_stat=plot_type_stat,
    include=include,
    std_range=std_range,
    std_unit=mf_units_print,
    check_sites=check_sites
)

Save plot:

In [ ]:
#output_path = 'PATH'

#fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

### 3. Spatial maps of country/region fluxes

In [ ]:
from fluxie.io import read_model_output
from fluxie.operators.select import slice_flux
from fluxie.config import set_model_labels
from fluxie.operators.flux_scale_by_sector_proportions import scale_by_sector_proportions

###################################
### edit variables in this block

# Comment or uncomment depending on CF4 plot (Supplement) or paneled species plot Fig 1
# species = ['cf4']
species = ['hfc23', 'ccl4','clch2ch2cl', 'c4f8', 'hcfc22', 'hcfc124', 'ch2cl2', 'chcl3', 'hfc134a','hfc32', 'hfc125', 'ccl2ccl2']


start_date = '2023-01-01'
end_date = '2026-03-01'

# models = experiments['industrial_base']
models = experiments['base_results']
# models = experiments['sensitivity']

period = 'yearly'
flux_units_print = 'kg km-2 yr-1'
get_labels_from_file = True
add_sites_to_flux = True
sector_file = None
sectors = ['total']
read_standard_run = False

###################################

species_list = species if isinstance(species, list) else [species]

ds_all_flux_scaled = {}
model_labels = {}

for sp in species_list:

    ds_all_flux = read_model_output(
        data_dir,
        "flux",
        sp,
        models,
        config_data,
        period=period,
        add_sites_to_flux=add_sites_to_flux,
        read_standard_run=read_standard_run,
    )

    ds_flux = slice_flux(
        ds_all_flux,
        config_data,
        start_date,
        end_date,
        species=sp,
        flux_units_print=flux_units_print,
    )

    labels_sp = set_model_labels(
        models,
        config_data,
        get_labels_from_file,
    )

    for m in ds_flux:

        key = f"{sp}__{m}"

        ds_all_flux_scaled[key] = ds_flux[m]
        model_labels[key] = f"{sp}\n{labels_sp.get(m, m)}"

if sector_file is not None:

    for key in ds_all_flux_scaled:

        sp = key.split("__")[0]

        ds_all_flux_scaled[key] = scale_by_sector_proportions(
            data_dir=data_dir,
            ds=ds_all_flux_scaled[key],
            species=sp,
            sectors=sectors,
            regions=None,
            country_flux_units_print=None,
            config_data=config_data,
            sector_file=sector_file,
            create_region_sector_totals=False,
        )

##### Prior and posterior fluxes comparison across models

In [ ]:
from fluxie.plots.flux_map import plot_flux_map

###################################
### edit variables in this block
region = [60, 100, 0, 40]
add_sites = True

species_markers = {
    "hfc23": [
        (77.8, 11.8),
        (72.6, 21.7),
        (72.9, 21.2),
        (76.8, 28.2),
    ],
    "hcfc22": [
        (77.8, 11.8),
        (72.6, 21.7),
        (72.9, 21.2),
        (76.8, 28.2),
    ], 
    "cf4": [
        (85.19, 20.85),
        (84.04, 21.82),
        (82.74, 22.39),
        (82.44, 24.22),
        (84.06, 21.74),
        (83.90, 21.53),
        (83.05, 24.20),
    ],
}

add_markers = species_markers

season = None # If specified, plot the seasonal mean (only valable for monthly data), options for 'DJF', 'MAM', 'JJA' and 'SON'

set_fluxlim = {"hcfc124": [0, 0.06], "hcfc22": [0, 8], "chcl3": [0, 50], "hfc134a": [0, 7], "hfc32": [0, 7], "hfc125": [0, 3], "hfc143a": [0, 0.6],
               "ccl4": [0, 7]} # for industrial
plot_inversion_grid_flux = True # If True, plot fluxes at inversion's resolution. If False, plot fluxes at prior's resolution.
sector = 'total'     # default 'total', otherwise chosen single sector plotted for all models
include_title_and_labels = True # if False, simplifys plots by removing title, y axis label, lat/lon ticks and some info from cbar label
marker_color=None
fallback_sites = ['MCOH', 'BCOB']
cmap = 'cividis'
only = 'posterior'
###################################

# spacings for 12 panel plot
fig = plot_flux_map(ds_all_flux_scaled, species, region, config_data, model_labels, season=season,
                    add_sites=add_sites, add_markers=add_markers, set_fluxlim=set_fluxlim, 
                    plot_inversion_grid_flux=plot_inversion_grid_flux,sector=sector,
                    include_title_and_labels=include_title_and_labels, marker_color=marker_color, 
                    fallback_sites=fallback_sites, cmap=cmap, only=only, ncols = 4,
                    row_spacing=0.3, col_spacing=-0.1, flux_cbar_offset=0.02, flux_label_offset= 0.1)

# # settings for 1 panel plot, CF4
# fig = plot_flux_map(ds_all_flux_scaled, species, region, config_data, model_labels, season=season,
#                     add_sites=add_sites, add_markers=add_markers, set_fluxlim=set_fluxlim, 
#                     plot_inversion_grid_flux=plot_inversion_grid_flux,sector=sector,
#                     include_title_and_labels=include_title_and_labels, marker_color=marker_color, 
#                     fallback_sites=fallback_sites, cmap=cmap, only=only, ncols = 1,
#                     row_spacing=0.3, col_spacing=-0.1, flux_cbar_offset=0.03, flux_label_offset= 0)



Save plot

In [ ]:
output_path = f'PATH'

fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2, dpi=300)